In [1]:
"""
Imports variables from urban_data_variables module.
"""
from collections import defaultdict
import pandas as pd
import geopandas as gpd
import re
from urban_data_variables import (
    districts,
    district_codes,
    schools,
    kindergartens,
    parking_spaces,
    public_transport_catchment_areas,
    parks_catchment_areas,
    healthcare_coverage
)
pd.set_option('display.max_rows', None)


In [ ]:
"""
Defines a dictionary mapping district names (keys) to locations from scraped files (values)
"""
district_to_locations = {
    "Sredets": [
        "Sofia, Tsentar"
    ],
    "Triaditsa": [
        "Sofia, Gotse Delchev",
        "Sofia, Ivan Vazov",
        "Sofia, Manastirski livadi",
        "Sofia, Meditsinska akademiya",
        "Sofia, Tsentar",
        "Sofia, zh.gr.Yuzhen park"
    ],
    "Lozenets": [
        "Sofia, Hladilnika",
        "Sofia, Krastova vada",
        "Sofia, Lozenets",
        "Sofia, PZ Hladilnika",
        "Sofia, Tsentar",
        "Sofia, zh.gr.Yuzhen park",
        "Sofia, zh.gr.Zoopark"
    ],
    "Vitosha": [
        "Sofia, Boyana",
        "Sofia, Dragalevtsi",
        "Sofia, Karpuzitsa",
        "Sofia, Knyazhevo",
        "Sofia, Krastova vada",
        "Sofia, Manastirski livadi",
        "Sofia, Pavlovo",
        "Sofia, Simeonovo",
        "Sofia, Vitosha",
        "Sofia, m-t Gardova glava",
        "Sofia, m-t Kinotsentara",
        "Sofia, v.z.Belovodski pat",
        "Sofia, v.z.Boyana",
        "Sofia, v.z.Cherniya kos",
        "Sofia, v.z.Gorna banya",
        "Sofia, v.z.Kiliite",
        "Sofia, v.z.Kinotsentara",
        "Sofia, v.z.Kinotsentara 3 chast",
        "Sofia, v.z.Simeonovo - Dragalevtsi"
    ],
    "Ovcha Kupel": [
        "Sofia, Bakston",
        "Sofia, Gorna banya",
        "Sofia, Ovcha kupel",
        "Sofia, Ovcha kupel 1",
        "Sofia, Ovcha kupel 2",
        "Sofia, Suhodol",
        "Sofia, v.z.Gorna banya",
        "Sofia, v.z.Kiliite"
    ],
    "Krasno Selo": [
        "Sofia, Bakston",
        "Sofia, Belite brezi",
        "Sofia, Borovo",
        "Sofia, Gradina",
        "Sofia, Hipodruma",
        "Sofia, Krasno selo",
        "Sofia, Lagera",
        "Sofia, Pavlovo",
        "Sofia, Slaviya",
        "Sofia, Strelbishte"
    ],
    "Krasna Polyana": [
        "Sofia, Krasna polyana 1",
        "Sofia, Krasna polyana 2",
        "Sofia, Krasna polyana 3",
        "Sofia, Razsadnika",
        "Sofia, Republika 2",
        "Sofia, Fakulteta",
        "Sofia, Zapaden park"
    ],
    "Vazrazhdane": [
        "Sofia, Tsentar",
        "Sofia, Zona B-18",
        "Sofia, Zona B-19",
        "Sofia, Zona B-5",
        "Sofia, Zona B-5-3"
    ],
    "Oborishte": [
        "Sofia, Doktorski pametnik",
        "Sofia, Oborishte",
        "Sofia, Tsentar",
        "Sofia, Yavorov"
    ],
    "Izgrev": [
        "Sofia, Dianabad",
        "Sofia, Izgrev",
        "Sofia, Iztok",
        "Sofia, NPZ Iztok"
    ],
    "Slatina": [
        "Sofia, 7-mi 11-ti kilometar",
        "Sofia, Geo Milev",
        "Sofia, Hristo Botev",
        "Sofia, NPZ Iskar",
        "Sofia, NPZ Iztok",
        "Sofia, Poligona",
        "Sofia, Reduta",
        "Sofia, Slatina",
        "Sofia, SPZ Slatina",
        "Sofia, Vrazhdebna",
        "Sofia, Yavorov"
    ],
    "Iskar": [
        "Sofia, Abdovitsa",
        "Sofia, Dimitar Milenkov",
        "Sofia, Druzhba 1",
        "Sofia, Druzhba 2",
        "Sofia, Eksperimentalen",
        "Sofia, NPZ Iskar"
    ],
    "Mladost": [
        "Sofia, 7-mi 11-ti kilometar",
        "Sofia, Gorublyane",
        "Sofia, Mladost 1",
        "Sofia, Mladost 1A",
        "Sofia, Mladost 2",
        "Sofia, Mladost 3",
        "Sofia, Mladost 4",
        "Sofia, Poligona",
        "Sofia, m-t Kambanite"
    ],
    "Studentski": [
        "Sofia, Darvenitsa",
        "Sofia, Malinova dolina",
        "Sofia, Musagenitsa",
        "Sofia, Studentski grad",
        "Sofia, v.z.Malinova dolina",
        "Sofia, v.z.Malinova dolina - Gerena"
    ],
    "Pancharevo": [
        "Sofia, m-t Detski grad",
        "Sofia, m-t Kambanite",
        "Sofia, m-t Shtarkelovo gnezdo",
        "Sofia, m-t yaz. Iskar",
        "Sofia, v.z.Amerikanski kolezh",
        "Sofia, v.z.Bunkera",
        "Sofia, v.z.Malinova dolina",
        "Sofia, v.z.Malinova dolina - Gerena",
        "Sofia, v.z.Vrana - German",
        "Sofia, v.z.Vrana - Lozen"
    ],
    "Kremikovtsi": [
        "Sofia, Botunets",
        "Sofia, Botunets 2",
        "Sofia, Chelopechene",
        "Sofia, Kremikovtsi",
        "Sofia, Seslavtsi"
    ],
    "Poduyane": [
        "Sofia, Hadzhi Dimitar",
        "Sofia, Levski",
        "Sofia, Levski G",
        "Sofia, Levski V",
        "Sofia, Malashevtsi",
        "Sofia, NPZ Hadzhi Dimitar",
        "Sofia, Poduyane",
        "Sofia, Suhata reka"
    ],
    "Serdika": [
        "Sofia, Banishora",
        "Sofia, Benkovski",
        "Sofia, Fondovi zhilishta",
        "Sofia, Malashevtsi",
        "Sofia, Orlandovtsi",
        "Sofia, Serdika",
        "Sofia, Voenna rampa"
    ],
    "Ilinden": [
        "Sofia, Gevgeliyski",
        "Sofia, Ilinden",
        "Sofia, Sveta Troitsa",
        "Sofia, Zaharna fabrika",
        "Sofia, Zapaden park"
    ],
    "Nadezhda": [
        "Sofia, Iliyantsi",
        "Sofia, Nadezhda 1",
        "Sofia, Nadezhda 2",
        "Sofia, Nadezhda 3",
        "Sofia, Nadezhda 4",
        "Sofia, PZ Iliyantsi",
        "Sofia, Svoboda",
        "Sofia, Tolstoy",
        "Sofia, Triagalnika",
        "Sofia, Voenna rampa"
    ],
    "Vrabnitsa": [
        "Sofia, Moderno predgradie",
        "Sofia, Obelya",
        "Sofia, Obelya 1",
        "Sofia, Obelya 2",
        "Sofia, PZ Iliyantsi",
        "Sofia, Vrabnitsa 1",
        "Sofia, Vrabnitsa 2",
        "Sofia, v.z.Lyulin"
    ],
    "Lyulin": [
        "Sofia, Filipovtsi",
        "Sofia, Lyulin 1",
        "Sofia, Lyulin 10",
        "Sofia, Lyulin 2",
        "Sofia, Lyulin 3",
        "Sofia, Lyulin 4",
        "Sofia, Lyulin 5",
        "Sofia, Lyulin 6",
        "Sofia, Lyulin 7",
        "Sofia, Lyulin 8",
        "Sofia, Lyulin 9",
        "Sofia, Lyulin - tsentar",
        "Sofia, Moderno predgradie",
        "Sofia, Republika 2",
        "Sofia, v.z.Lyulin"
    ],
    "Novi Iskar": [
        "Sofia, Trebich"
    ],
    "Bankya": []
}


In [ ]:
"""
Reads scraped files, reverses dictionary for further mapping.
"""
sofia_for_rent_scraped = pd.read_csv(r'D:\Data Analysis\5_Projects\sofia_housing_districts_analysis\data\housing_listings_scraped\sofia_for_rent_scraped.csv')
sofia_for_sale_scraped = pd.read_csv(r'D:\Data Analysis\5_Projects\sofia_housing_districts_analysis\data\housing_listings_scraped\sofia_for_sale_scraped.csv')

loc_to_districts = defaultdict(list)
for district, locs in district_to_locations.items():
    for loc in locs:
        loc_to_districts[loc].append(district)


In [ ]:
"""
Maps districts for sofia_for_rent_scraped file into a pd.dataframe variable and creates a bridge_rent_listings_ad_id_district file.
"""
sofia_for_rent_scraped['districts'] = sofia_for_rent_scraped['location'].map(loc_to_districts)
sofia_for_rent_scraped['districts'] = sofia_for_rent_scraped['location'].map(
    lambda loc: ', '.join(loc_to_districts[loc]) if loc in loc_to_districts else None
)

bridge_rent_listings = (
    sofia_for_rent_scraped[['ad_id', 'districts']]   
    .explode('districts')                            
    .dropna(subset=['districts'])
    .rename(columns={'districts': 'district'})
)

bridge_rent_listings.to_csv(r'D:\Data Analysis\5_Projects\sofia_housing_districts_analysis\results\intermediate\bridge_rent_listings_ad_id_district.csv')

In [ ]:
"""
Maps districts for sofia_for_sale_scraped file into a pd.dataframe variable and creates a bridge_sale_listings_ad_id_district file.
"""
sofia_for_sale_scraped['districts'] = sofia_for_sale_scraped['location'].map(loc_to_districts)
sofia_for_sale_scraped['districts'] = sofia_for_sale_scraped['location'].map(
    lambda loc: ', '.join(loc_to_districts[loc]) if loc in loc_to_districts else None
)

bridge_sale_listings = (
    sofia_for_sale_scraped[['ad_id', 'districts']]   
    .explode('districts')                            
    .dropna(subset=['districts'])
    .rename(columns={'districts': 'district'})
)

bridge_sale_listings.to_csv(r'D:\Data Analysis\5_Projects\sofia_housing_districts_analysis\results\intermediate\bridge_sale_listings_ad_id_district.csv')